## 🎯 Learning Objectives
* Understand the necessity and benefits of multi-step retrieval in advanced RAG systems.
* Learn how to design and implement multi-step retrieval graphs using LangGraph.
* Identify key components of a multi-step retrieval graph, including state management, nodes, and conditional edges.
* Analyze the performance trade-offs and typical use cases for multi-step retrieval graphs.


## Multi-step Retrieval Graphs in LangGraph

In the evolving landscape of Retrieval-Augmented Generation (RAG), simple, single-pass retrieval often falls short when confronted with complex, ambiguous, or multi-faceted queries. Imagine a seasoned research assistant: they don't just grab the first relevant document and call it a day. Instead, they might perform an initial broad search, skim the results, identify new keywords or sub-questions, refine their search strategy, consult multiple specialized databases, and only then synthesize a comprehensive answer. This iterative, adaptive process is the essence of **multi-step retrieval**.

### Why Multi-step Retrieval?

Traditional RAG systems typically follow a linear path: query -> retrieve -> generate. While effective for straightforward questions, this approach struggles with:

1.  **Ambiguity**: Initial search results might be too broad or irrelevant if the query is vague.
2.  **Complexity**: A single query might implicitly require information from multiple distinct domains or perspectives.
3.  **Evolving Information Needs**: The relevance of information can change as new facts are uncovered.
4.  **Hallucination Mitigation**: By iteratively refining retrieval, the system can gather more precise evidence, reducing the likelihood of generating incorrect information.
5.  **Self-Correction**: If initial retrieval yields poor results, the system can detect this and attempt a different strategy.

### LangGraph as the Orchestrator

LangGraph, built on top of LangChain, provides the perfect framework for orchestrating these complex, multi-step retrieval processes. It allows us to define RAG as a **state machine**, where each step (or "node") performs a specific action (e.g., initial retrieval, query rewriting, re-ranking, answer synthesis) and the transitions between these steps ("edges") are governed by conditional logic. This enables dynamic, adaptive workflows that can mimic the sophisticated reasoning of a human researcher.

**Core Components of a Multi-step Retrieval Graph:**

*   **Graph State**: A shared object that holds all relevant information throughout the graph's execution, such as the original query, retrieved documents, refined queries, intermediate analyses, and the final answer. This state is passed between nodes.
*   **Nodes**: Functions or runnable LangChain components that perform specific tasks. Examples include:
    *   `retrieve_documents`: Fetches documents based on a query.
    *   `rewrite_query`: Uses an LLM to rephrase or expand the query based on initial context or lack of relevant results.
    *   `evaluate_retrieval`: Assesses the quality of retrieved documents (e.g., using an LLM or a re-ranker) and decides if more retrieval is needed.
    *   `generate_sub_queries`: Breaks down a complex query into simpler sub-queries.
    *   `synthesize_answer`: Generates the final response based on all gathered information.
*   **Edges**: Define the flow between nodes. Crucially, LangGraph supports **conditional edges**, allowing the graph to branch based on the output of a node or the current state. For instance, after evaluating retrieval, the graph might decide to either `rewrite_query` and `retrieve_documents` again, or proceed directly to `synthesize_answer`.

### An Illustrative Example: Iterative Query Refinement

Consider a scenario where a user asks: "What are the latest advancements in AI for drug discovery?" A simple search might return general articles. A multi-step graph could:

1.  **Initial Retrieval**: Search for "AI drug discovery advancements".
2.  **Analysis & Refinement**: An LLM reviews the initial results. If they are too broad or outdated, it might suggest refining the query to "recent deep learning applications drug discovery" or "AI-driven drug target identification 2024-2026".
3.  **Refined Retrieval**: Perform a new search with the refined query.
4.  **Synthesis**: Combine information from both retrieval steps to form a comprehensive answer.

This dynamic approach ensures that the RAG system is not just retrieving, but actively *reasoning* about its retrieval process, leading to more accurate, relevant, and robust responses.


In [ ]:
import os
from typing import List, Dict, TypedDict, Union
from langchain_core.documents import Document
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END

# Ensure you have your OpenAI API key set as an environment variable
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

# --- 1. Define the Graph State ---
# This defines the object that is passed between nodes in the graph.
class GraphState(TypedDict):
    """Represents the state of our graph. """
    query: str  # The user's original or refined query
    documents: List[Document]  # List of retrieved documents
    refined_query: str  # A potentially refined query
    generation: str  # The final generated answer
    steps: List[str] # To track the execution path

# --- 2. Mock Data and Retriever (for demonstration) ---
# In a real application, this would be a robust vector store like Chroma, Weaviate, or Pinecone.
doc_corpus = [
    Document(page_content="Quantum machine learning (QML) is an emerging field that explores the intersection of quantum computing and machine learning. Challenges include data encoding, limited qubit coherence, and error correction. Deployment often requires specialized hardware and software stacks.", metadata={"source": "QML_Overview"}),
    Document(page_content="Classical machine learning (CML) model deployment involves MLOps practices like containerization (Docker, Kubernetes), CI/CD pipelines, model monitoring, and scaling on cloud platforms (AWS, Azure, GCP). Key challenges are data drift, model decay, and ensuring low-latency inference.", metadata={"source": "CML_Deployment"}),
    Document(page_content="Recent breakthroughs in quantum algorithms show promise for accelerating certain ML tasks, but practical applications are still in early research stages. Hardware limitations remain a significant bottleneck for widespread adoption.", metadata={"source": "Quantum_Breakthroughs"}),
    Document(page_content="The MLOps ecosystem for classical models is mature, offering tools for automated deployment, versioning, and governance. However, integrating new research models into production still poses challenges due to varying frameworks and dependencies.", metadata={"source": "MLOps_Ecosystem"}),
    Document(page_content="Hybrid quantum-classical algorithms are being developed to leverage the strengths of both paradigms, aiming to mitigate some of the current quantum hardware limitations for ML tasks.", metadata={"source": "Hybrid_QML"})
]

def mock_retriever(query: str, k: int = 3) -> List[Document]:
    """A simple mock retriever that simulates document retrieval based on keyword matching."""
    print(f"--- Retrieving documents for query: '{query}' ---")
    results = []
    query_lower = query.lower()
    for doc in doc_corpus:
        if any(keyword in doc.page_content.lower() for keyword in query_lower.split()):
            results.append(doc)
    # Sort by relevance (simple heuristic: more keywords match = more relevant)
    results.sort(key=lambda x: sum(1 for keyword in query_lower.split() if keyword in x.page_content.lower()), reverse=True)
    return results[:k]

# --- 3. Initialize LLM ---
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)

# --- 4. Define Graph Nodes ---

def retrieve_initial(state: GraphState) -> GraphState:
    """Node: Performs initial document retrieval based on the original query."""
    print("---NODE: INITIAL RETRIEVAL---")
    query = state["query"]
    documents = mock_retriever(query)
    return {"documents": documents, "steps": state.get("steps", []) + ["initial_retrieval"]}

def analyze_and_refine_query(state: GraphState) -> GraphState:
    """Node: Analyzes retrieved documents and decides if the query needs refinement or more retrieval."""
    print("---NODE: ANALYZE AND REFINE QUERY---")
    query = state["query"]
    documents = state["documents"]

    # LLM prompt to analyze documents and decide on refinement
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an expert research assistant. Analyze the provided documents in relation to the user's query. Determine if the documents are sufficient to answer the query directly, or if the query needs to be refined for a more targeted search. If refinement is needed, suggest a new, more specific query. If sufficient, state 'NO_REFINEMENT'.
        
        Original Query: {query}
        Retrieved Documents:
        {documents}
        
        Based on the above, should the query be refined? If yes, provide the refined query. If no, state 'NO_REFINEMENT'.
        Example refined query: 'specific aspects of X related to Y'
        Example no refinement: 'NO_REFINEMENT'
        
        Refined Query (or NO_REFINEMENT):"),
        ("human", "{query}")
    ])

    chain = prompt | llm | StrOutputParser()
    
    # Format documents for the prompt
    docs_content = "\n---\n".join([doc.page_content for doc in documents])
    
    refined_output = chain.invoke({"query": query, "documents": docs_content})
    
    refined_query = refined_output.strip()
    print(f"Refinement LLM output: {refined_query}")

    if refined_query == "NO_REFINEMENT":
        return {"refined_query": query, "steps": state.get("steps", []) + ["analyze_no_refinement"]}
    else:
        return {"refined_query": refined_query, "steps": state.get("steps", []) + ["analyze_and_refine"]}

def retrieve_refined(state: GraphState) -> GraphState:
    """Node: Performs document retrieval using the refined query."""
    print("---NODE: REFINED RETRIEVAL---")
    refined_query = state["refined_query"]
    # Combine existing documents with new ones, avoiding duplicates
    new_documents = mock_retriever(refined_query)
    existing_docs_content = {doc.page_content for doc in state["documents"]}
    combined_documents = state["documents"] + [doc for doc in new_documents if doc.page_content not in existing_docs_content]
    return {"documents": combined_documents, "steps": state.get("steps", []) + ["refined_retrieval"]}

def generate_answer(state: GraphState) -> GraphState:
    """Node: Generates the final answer based on all retrieved documents."""
    print("---NODE: GENERATE ANSWER---")
    query = state["query"]
    documents = state["documents"]

    # LLM prompt to synthesize answer
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful AI assistant. Based on the following documents, provide a comprehensive and concise answer to the user's query. Cite your sources by their metadata 'source' if available. If the documents do not contain enough information, state that.
        
        Query: {query}
        Documents:
        {documents}
        
        Answer:"),
        ("human", "{query}")
    ])

    chain = prompt | llm | StrOutputParser()
    
    docs_content = "\n---\n".join([f"Source: {doc.metadata.get('source', 'N/A')}\nContent: {doc.page_content}" for doc in documents])
    
    generation = chain.invoke({"query": query, "documents": docs_content})
    return {"generation": generation, "steps": state.get("steps", []) + ["generate_answer"]}

# --- 5. Define Conditional Edges ---

def decide_to_retrieve_more(state: GraphState) -> str:
    """Conditional edge: Decides whether to perform refined retrieval or generate an answer."""
    print("---DECISION: RETRIEVE MORE?---")
    if state["refined_query"] == state["query"]:
        print("Decision: No refinement needed, proceeding to generate answer.")
        return "generate_answer"
    else:
        print("Decision: Refined query available, proceeding to refined retrieval.")
        return "retrieve_refined"

# --- 6. Build the LangGraph ---

workflow = StateGraph(GraphState)

# Add nodes
workflow.add_node("retrieve_initial", retrieve_initial)
workflow.add_node("analyze_and_refine_query", analyze_and_refine_query)
workflow.add_node("retrieve_refined", retrieve_refined)
workflow.add_node("generate_answer", generate_answer)

# Set entry point
workflow.set_entry_point("retrieve_initial")

# Add edges
workflow.add_edge("retrieve_initial", "analyze_and_refine_query")
workflow.add_conditional_edges(
    "analyze_and_refine_query",
    decide_to_retrieve_more, # This function determines the next node
    {
        "retrieve_refined": "retrieve_refined",
        "generate_answer": "generate_answer"
    }
)
workflow.add_edge("retrieve_refined", "generate_answer")

# Set exit point
workflow.set_finish_point("generate_answer")

# Compile the graph
app = workflow.compile()

# --- 7. Run the Graph with Example Queries ---

print("\n==================================================")
print("RUN 1: Query requiring refinement")
print("==================================================")
initial_state_1 = {"query": "What are the main challenges in deploying quantum machine learning models, and how do they compare to classical ML deployment issues?", "documents": [], "refined_query": "", "generation": "", "steps": []}
for s in app.stream(initial_state_1):
    print(s)
    print("----")

print("\nFinal Answer 1:")
print(s[END]["generation"])
print(f"Execution Path 1: {s[END]['steps']}")

print("\n==================================================")
print("RUN 2: Query that might not need refinement")
print("==================================================")
initial_state_2 = {"query": "Tell me about hybrid quantum-classical algorithms.", "documents": [], "refined_query": "", "generation": "", "steps": []}
for s in app.stream(initial_state_2):
    print(s)
    print("----")

print("\nFinal Answer 2:")
print(s[END]["generation"])
print(f"Execution Path 2: {s[END]['steps']}")


### Interpreting the Code Output and Performance Trade-offs

The code above demonstrates a basic multi-step retrieval graph. Let's break down its output and implications:

**Interpreting the Output:**

When you run the code, you'll observe the `print` statements from each node and decision point, clearly illustrating the graph's execution flow. For `RUN 1` (a complex query), you should see:

1.  `---NODE: INITIAL RETRIEVAL---`: The graph starts by fetching documents for the original query.
2.  `---NODE: ANALYZE AND REFINE QUERY---`: An LLM reviews these initial documents. For a complex query, it's likely to determine that the query needs refinement.
3.  `---DECISION: RETRIEVE MORE?---`: The conditional edge function `decide_to_retrieve_more` is called. Since the LLM suggested a refined query, it directs the flow to `retrieve_refined`.
4.  `---NODE: REFINED RETRIEVAL---`: A second retrieval step occurs, using the LLM-generated refined query. This often brings in more targeted or complementary information.
5.  `---NODE: GENERATE ANSWER---`: Finally, the LLM synthesizes an answer using *all* collected documents (from both initial and refined retrieval).

For `RUN 2` (a simpler, more direct query), the `analyze_and_refine_query` node might output `NO_REFINEMENT`. In this case, the `decide_to_retrieve_more` function would directly route to `generate_answer`, skipping the `retrieve_refined` step. This showcases the adaptive nature of the graph.

The `steps` list in the final state provides a clear audit trail of the path taken through the graph.

**Performance Trade-offs:**

Multi-step retrieval graphs offer significant advantages but come with their own set of considerations:

**Advantages:**

*   **Enhanced Accuracy and Relevance**: By iteratively refining queries and gathering more targeted information, these systems can provide much more precise and relevant answers, especially for complex or ambiguous questions.
*   **Reduced Hallucination**: Access to a broader and more refined set of evidence reduces the LLM's reliance on its internal knowledge, thereby lowering the risk of generating incorrect or fabricated information.
*   **Improved Robustness**: The ability to self-correct or adapt retrieval strategies makes the system more resilient to poorly formulated initial queries or sparse initial search results.
*   **Better Handling of Complex Information Needs**: Can break down multi-part questions into sub-queries, mimicking human reasoning processes.

**Disadvantages:**

*   **Increased Latency**: Each additional retrieval step and LLM call adds to the overall processing time. A multi-step graph will inherently be slower than a single-pass RAG system.
*   **Higher Computational Cost**: More LLM calls and potentially more extensive retrieval operations translate to higher API costs and computational resource usage.
*   **Design Complexity**: Building and debugging multi-step graphs requires careful thought about state management, node responsibilities, and conditional logic. Prompt engineering for the LLM-driven decision nodes (like `analyze_and_refine_query`) becomes critical.
*   **Potential for Over-refinement**: In some cases, an LLM might unnecessarily refine a query that was already sufficient, leading to wasted cycles and increased latency without a proportional gain in quality.

### Typical Use Cases

Multi-step retrieval graphs are particularly well-suited for applications where accuracy, comprehensiveness, and robustness are paramount, and where the queries are often complex or require deep understanding:

*   **Advanced Research Assistants**: Systems that help users explore complex topics, summarize findings from multiple sources, or answer nuanced questions in scientific, legal, or medical domains.
*   **Customer Support Bots for Technical Products**: Handling intricate troubleshooting scenarios that might require iterative information gathering based on user input and system diagnostics.
*   **Legal and Compliance Q&A**: Answering questions that require cross-referencing multiple legal documents, statutes, and case precedents.
*   **Competitive Intelligence**: Analyzing market trends, competitor strategies, or technological landscapes by synthesizing information from diverse data sources.
*   **Self-Correcting RAG Systems**: Implementing mechanisms like Self-RAG or Corrective RAG where the system evaluates its own retrieval and generation quality and takes corrective actions if needed.

By leveraging LangGraph, AI search engineers can move beyond static RAG pipelines to build truly intelligent, adaptive, and highly effective information retrieval and generation systems.


### Resources

*   **LangGraph Documentation**: The official source for understanding LangGraph's concepts and API. [https://langchain-ai.github.io/langgraph/](https://langchain-ai.github.io/langgraph/)
*   **LangChain RAG Documentation**: Comprehensive guides on building RAG applications with LangChain. [https://python.langchain.com/docs/use_cases/question_answering/](https://python.langchain.com/docs/use_cases/question_answering/)
*   **OpenAI API Documentation**: For integrating powerful LLMs like GPT-4o-mini. [https://platform.openai.com/docs/api-reference](https://platform.openai.com/docs/api-reference)
*   **Research Paper: Self-RAG: Learning to Retrieve, Generate, and Critique through Self-Reflection**: A foundational paper on agentic RAG that inspires multi-step approaches. [https://arxiv.org/abs/2310.11511](https://arxiv.org/abs/2310.11511)
*   **Research Paper: Corrective RAG (CRAG)**: Another relevant paper discussing self-correction in RAG. [https://arxiv.org/abs/2401.15884](https://arxiv.org/abs/2401.15884)
